<a href="https://colab.research.google.com/github/camlet0630/ESG_ranking_system/blob/main/ESG_model_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set

In [ ]:
pip install transformers

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from tqdm import tqdm
import random
from collections import Counter
from IPython.display import display


import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizerFast, AutoModel
from transformers import AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold

In [ ]:
device = torch.device('cuda', 0) if torch.cuda.is_available() else 'cpu'
print(device)

# BERT Tokenizer: 使用 'bert-base-chinese' 版本
tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')

# BERT Model: 使用 'bert-base-chinese' 版本
bert = AutoModel.from_pretrained('bert-base-chinese').to(device)

In [ ]:
# config
config = {"embedding_path": "/content/drive/MyDrive/embedding_tensor.pt",
          "seed": 1006,
          "learning_rate": 2e-5,
          "batch_size": 256, #128 #256
          "epochs": 300}

In [ ]:
def set_random_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)

set_random_seed(config["seed"])

## Data preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
raw_data = pd.read_csv("/content/drive/MyDrive/news.csv")
raw_data.head(5)

In [ ]:
num_of_content = len(raw_data)
num_of_content

In [ ]:
# provided by chatGPT
"""
import pandas as pd

# 創建一個 DataFrame
data = {'Name': ['Alice', 'Bob', 'Charlie', 'David'],
        'Age': [25, 30, 35, 40],
        'City': ['New York', 'San Francisco', 'Los Angeles', 'Chicago']}

df = pd.DataFrame(data)

# 顯示原始 DataFrame
print("原始 DataFrame:")
print(df)

# 在 for loop 中根據條件刪除行
for index, row in df.iterrows():
    # 假設你想根據條件刪除年齡大於 30 的行
    if row['Age'] > 30:
        df = df.drop(index)

# 顯示刪除行後的 DataFrame
print("\n刪除符合條件的行後的 DataFrame:")
print(df)
"""

In [ ]:
# 在 for loop 中根據條件刪除行
# 刪除沒有 content 的 row
for index, row in raw_data.iterrows():
  if type(raw_data["Content"][index]) != str:
    raw_data = raw_data.drop(index)

In [ ]:
num_of_content = len(raw_data)
num_of_content

In [ ]:
type(raw_data)

In [ ]:
# 將每個段落都加上標題
"""
for i in range(1, len(raw_data)):
  if type(raw_data["Name"][i]) != str:
    raw_data["Name"][i] = raw_data["Name"][i-1]
"""

now = "now"
for index, row in raw_data.iterrows():
  if type(raw_data["Name"][index]) == str:
    now = raw_data["Name"][index]
  if type(raw_data["Name"][index]) != str:
    raw_data["Name"][index] = now

raw_data

### 將 label data 轉為 "E", "S", "G"

In [ ]:
# 創建欄位
lis = [0] * num_of_content
raw_data["E"] = lis
raw_data["S"] = lis
raw_data["G"] = lis

In [ ]:
# provided by chatGPT
"""
import re

def check_substrings(main_string, substrings):
    for substring in substrings:
        pattern = re.compile(re.escape(substring))
        if pattern.search(main_string):
            return True
    return False

# 範例
main_str = "這個城市有空氣汙染和水汙染"
sub_strs = ["空氣汙染", "水汙染", "噪音汙染"]

result = check_substrings(main_str, sub_strs)
print(result)

if result:
    print(f"{main_str} 中包含完整子字串中的至少一個")
else:
    print(f"{main_str} 中不包含完整子字串中的任何一個")
"""

In [ ]:
# 檢查 tag 中是否包含字串

import re

def check_substrings(main_string, substrings):
    for substring in substrings:
        pattern = re.compile(re.escape(substring))
        if pattern.search(main_string):
            return True
    return False

def is_E(main_string):
  substrings = ["空氣污染", "能源管理", "燃料管理", "產品包裝", "生物多樣性", "溫室氣體排放", "水及污染管理", "廢棄物處理"]
  #substrings = ["空氣污染"]
  return check_substrings(main_string, substrings)

def is_S(main_string):
  substrings = ["人權", "社區關係", "客戶福利", "勞工關係", "薪酬與福利", "多樣性與共融", "僱員健康安全"]
  return check_substrings(main_string, substrings)

def is_G(main_string):
  substrings = ["商業倫理", "物料採購", "競爭行為", "激勵措施", "供應鏈管理", "系統化管理", "意外及安全"]
  return check_substrings(main_string, substrings)

In [ ]:
for index, row in raw_data.iterrows():
  if type(raw_data["Tag01"][index]) == str:
    if is_E(raw_data["Tag01"][index]):
      raw_data["E"][index] = 1
    if is_S(raw_data["Tag01"][index]):
      raw_data["S"][index] = 1
    if is_G(raw_data["Tag01"][index]):
      raw_data["G"][index] = 1

  if type(raw_data["Tag02"][index]) == str:
    if is_E(raw_data["Tag02"][index]):
      raw_data["E"][index] = 1
    if is_S(raw_data["Tag02"][index]):
      raw_data["S"][index] = 1
    if is_G(raw_data["Tag02"][index]):
      raw_data["G"][index] = 1

  if type(raw_data["Tag03"][index]) == str:
    if is_E(raw_data["Tag03"][index]):
      raw_data["E"][index] = 1
    if is_S(raw_data["Tag03"][index]):
      raw_data["S"][index] = 1
    if is_G(raw_data["Tag03"][index]):
      raw_data["G"][index] = 1

In [ ]:
# print(raw_data.head(40))

In [ ]:
news_articles = raw_data["Name"]+raw_data["Content"]
label_data = raw_data.iloc[:,-3:]

In [ ]:
print(news_articles)
print(label_data)

In [ ]:
type(news_articles)

In [ ]:
data = news_articles + label_data

## Evaluation Metrics

In [ ]:
# 函數: 計算 Accuracy
def ACCscore_m_label(y_true, pred):
    accuracy_l = [ans.all() for ans in (pred == y_true)]
    accuracy = np.array(accuracy_l).mean()
    return accuracy


# 函數: 計算 Precision, Recall
def PRscore_m_label(y_true, pred):
    hit_matrix = np.zeros_like(pred)
    hit_matrix[np.where((pred == y_true) & (y_true > 0))] = 1
    tp = hit_matrix.sum(axis=1)
    pred_sum = pred.sum(axis=1)
    true_sum = y_true.sum(axis=1)
    precision_l = []
    recall_l = []
    for ix in range(tp.shape[0]):
        precision_score = (1.0 if true_sum[ix] == 0 else 0.0) if pred_sum[ix] == 0 else tp[ix]/pred_sum[ix]
        recall_score = (1.0 if pred_sum[ix] == 0 else 0.0) if true_sum[ix] == 0 else tp[ix]/true_sum[ix]
        precision_l.append(precision_score)
        recall_l.append(recall_score)
    precision = np.array(precision_l).mean()
    recall = np.array(recall_l).mean()
    return precision, recall


# 函數: 計算 F1-score
def f1(precision, recall):
    f1_score = 0
    if (precision + recall) !=0:
        f1_score = (2 * precision * recall) / (precision + recall)
    return f1_score


# 函數: 一次同時計算所有評估指標
def score(y_true, pred):
    accuracy = ACCscore_m_label(y_true, pred)
    precision, recall = PRscore_m_label(y_true, pred)
    f1_score = f1(precision, recall)
    return accuracy, precision, recall, f1_score

## Modeling

### BERT Embedding

In [ ]:
def get_embedding_tensor(context_ary):

    embedding_tensor = torch.ones((context_ary.shape[0], 768))


    for i in range(context_ary.shape[0]):

        context = context_ary[i]
        bert_input = tokenizer(context, padding='max_length', max_length=512,
                               truncation=True, return_tensors="pt")


        attention_mask = bert_input['attention_mask'].to(device)
        input_id = bert_input['input_ids'].squeeze(1).to(device)
        final_inputs = {'input_ids': input_id, 'attention_mask': attention_mask}
        outputs = bert(**final_inputs)

        ## Only the embedding vector of [CLS] is taken
        pooler_output = outputs.last_hidden_state[0][0].reshape(768)
        embedding_tensor[i] = pooler_output.detach().cpu()

        gc.collect()
        torch.cuda.empty_cache()

    return embedding_tensor

In [ ]:
"""
# 計算每個樣本各自的 BERT Embedding
context_ary = news_articles.values
embedding_tensor = get_embedding_tensor(context_ary)

# 儲存 BERT Embedding Tensor
torch.save(embedding_tensor, config["embedding_path"])
"""

In [ ]:
# 載入 BERT Embedding Tensor
embedding_tensor = torch.load(config["embedding_path"])
print(embedding_tensor.shape)

### Calculate Class Weight

In [ ]:
# Function: Class weight calculation
def get_class_weight(df, labels):
    # 計算每個類別的「正樣本數量」，儲存成 class_pos_count 列表
    class_pos_count = np.zeros(len(labels))
    # 計算每個類別的「負樣本數量」，儲存成 class_neg_count 列表
    class_neg_count = np.zeros(len(labels))
    # 遞迴
    for i in range(len(labels)):
        label = labels[i]
        positive, negative = df[label].value_counts()[1.0], df[label].value_counts()[0.0]
        class_pos_count[i], class_neg_count[i] = positive, negative
    # 計算「正樣本權重」
    class_pos_weight = np.ones_like(class_pos_count)
    for cdx, (pos_count, neg_count) in enumerate(zip(class_pos_count, class_neg_count)):
        total_count = pos_count + neg_count
        postive_weights = total_count/(2*pos_count)
        negtive_weights = total_count/(2*neg_count)
        class_pos_weight[cdx] = (postive_weights/negtive_weights)
    return torch.as_tensor(class_pos_weight, dtype=torch.float)

### Print out Class Weight

In [ ]:
def print_class_weight(task_label):
    print("<<各類別資料數>>")
    print(task_label.sum())
    label_list = task_label.columns
    class_weight = get_class_weight(task_label, label_list)
    print()
    print("<<目標變數與權重對應>>")
    print(f"目標變數共有 {len(label_list)} 類")
    for c, w in zip(label_list, class_weight):
        print(f"{c}: {round(float(w), 2)}")

### Build Class: Dataset

In [ ]:
class Dataset(Dataset):
    def __init__(self, embeddings, label_list):
        self.embeddings = embeddings
        self.labels = label_list.to_numpy()

    def __getLabels__(self):
        return (self.labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

In [ ]:
class Classifier(nn.Module):
    def __init__(self, output_size=3, dropout_rate=0.85):
        super(Classifier, self).__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear_1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(256, 64)
        self.linear_3 = nn.Linear(64, output_size)


    def forward(self, embeddings):
        output = self.linear_1(embeddings)
        output = self.relu(output)
        output = self.dropout(output)
        output = self.linear_2(output)
        output = self.relu(output)
        #output = self.dropout(output)
        output = self.linear_3(output)

        return output

### Model Architecture
- 實現一個神經網絡模型。該模型包含了兩層隱藏層架構並使用 dropout 機制

In [ ]:
"""
# milti task
class Classifier(nn.Module):
    def __init__(self, output_size_1=3, output_size_2=22, dropout_rate=0.5):
        super(Classifier, self).__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear_1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(256, 64)
        self.linear_3 = nn.Linear(64, output_size_1)
        self.linear_4 = nn.Linear(64, 32)  # yellow_1
        self.linear_5 = nn.Linear(32+3, output_size_2)

    def forward(self, embeddings):
        output = self.linear_1(embeddings)
        output = self.relu(output)
        output = self.linear_2(output)
        shared_output = self.relu(output) # black block
        output = self.dropout(output)
        output_1 = self.linear_3(shared_output)
        output_2 = self.linear_4(share_output)
        output_cat = torch.cat((output_1, output_2),axis=1)
        output_cat = self.relu(output_cat)
        output_2 = self.linear_5(output_cat)
        return output_1, output_2
"""

In [ ]:
"""
# including teacher forcing
# share layer 的層數
# 各自任務的層數 e.g. concate 完可以再接
# dropout
class Classifier(nn.Module):
    def __init__(self, output_size_1=3, output_size_2=22, dropout_rate=0.5):
        super(Classifier, self).__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear_1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(256, 64)
        self.linear_3 = nn.Linear(64, output_size_1)
        self.linear_4 = nn.Linear(64, 32)  # yellow_1 [(output 的部分)可以嘗試只接layer1]
        self.linear_5 = nn.Linear(32+3, output_size_2)

    def forward(self, embeddings, y1_true, alpha):  # y1_true = E/S/G
        output = self.linear_1(embeddings)
        output = self.relu(output)
        output = self.linear_2(output)
        shared_output = self.relu(output) # black block
        output = self.dropout(output)
        output_1 = self.linear_3(output)
        output_2 = self.linear_4(share_output)
        output_2 = self.relu(output_2)
        rept_ESG = alpha*y1_true + (1-alpha)*output_1
        output_cat = torch.cat((rept_ESG, output_2),axis=1)
        # output_cat = self.dropout(output_cat)
        output_2 = self.linear_5(output_cat)
        return output_1, output_2
"""

In [ ]:
"""
# positive/negative
class Classifier(nn.Module):
    def __init__(self, output_size_1=3, output_size_2=22, dropout_rate=0.5):
        super(Classifier, self).__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear_1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.linear_2 = nn.Linear(256, 64)
        self.linear_3 = nn.Linear(64, output_size_1)
        self.linear_4 = nn.Linear(64, 32)  # yellow_1 [(output 的部分)可以嘗試只接layer1]
        self.linear_5 = nn.Linear(32+3, output_size_2)
        self.output_layers = nn.ModuleList([nn.Sequential(nn.Linear(32+3, 3), nn.Softmax) for _ in range(output_size_2)])

    def forward(self, embeddings, y1_true, alpha):  # y1_true = E/S/G
        output = self.linear_1(embeddings)
        output = self.relu(output)
        output = self.linear_2(output)
        shared_output = self.relu(output) # black block
        shared_output = self.dropout(shared_output)
        output_1 = self.linear_3(shared_output)
        output_2 = self.linear_4(share_output)
        output_2 = self.relu(output_2)
        rept_ESG = alpha*y1_true + (1-alpha)*output_1
        output_cat = torch.cat((rept_ESG, output_2),axis=1)
        # output_cat = self.dropout(output_cat)
        output_2 = [output_layer(output_cat) for output_layer in self.output_layers]
        return output_1, output_2
"""

### Function for model training

In [ ]:
def train(model, class_weight, train_set, val_set, patience=1000):
    train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)
    # Loss Function
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weight)

    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

    # Putting the model on the GPU to run
    model = model.to(device)
    criterion = criterion.to(device)

    val_best_f1 = float(-np.inf)
    iteration_num = 0
    epoch_num = 0

    epochs = config["epochs"]

    while True:
        if iteration_num >= patience:
            break

        # Train and save training results
        loss_list = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        f1_list = []

        # Set the model to the "training" state (Layers such as Dropout will be triggered only)
        model.train()

        for train_embeddings, train_labels in tqdm(train_loader):

            optimizer.zero_grad()

            train_embeddings = train_embeddings.to(device)
            train_labels = train_labels.to(device)

            output = model(train_embeddings)

            batch_loss = criterion(output, train_labels.float())

            # Back propagation
            batch_loss.backward()
            optimizer.step()

            output = output.cpu().detach()
            pred = (output > 0).cpu()
            y_true = (train_labels > 0).cpu()

            # Calculate the performance of each batch and save it
            loss = batch_loss.item()
            accuracy, precision, recall, f1_score= score(y_true, pred)

            loss_list.append(loss)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            recall_list.append(recall)
            f1_list.append(f1_score)

        # Calculate the performance of each epoch and save it
        train_loss = np.array(loss_list).mean()
        train_accuracy = np.array(accuracy_list).mean()
        train_precision = np.array(precision_list).mean()
        train_recall = np.array(recall_list).mean()
        train_f1 = np.array(f1_list).mean()


        # Evaluate the current model performance using validate dataset
        loss_list = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        pred_list = []
        f1_list = []

        with torch.no_grad():
            model.eval()
            for val_embeddings, val_labels in tqdm(val_loader):

                val_embeddings = val_embeddings.to(device)
                val_labels = val_labels.to(device)

                output = model(val_embeddings)

                batch_loss = criterion(output, val_labels.float())

                output = output.cpu()
                pred = (output > 0).cpu()
                sum_pred = torch.sum(pred, dim=1)
                non_label_case_idx = (sum_pred < 1).nonzero()

                if non_label_case_idx.shape[0] != 0:
                    non_label_case = output[non_label_case_idx]
                    max_col = torch.argmax(non_label_case, dim=-1)
                    raw = non_label_case_idx.reshape(-1,1)
                    pred_np = pred.numpy()
                    for idx in range(raw.shape[0]):
                        pred_np[raw[idx]][max_col[idx]] = True
                    pred = torch.from_numpy(pred_np)
                pred_list.extend(pred.numpy().astype('float32'))
                y_true = (val_labels > 0).cpu()

                # Calculate the performance of each batch and save it
                loss = batch_loss.item()
                accuracy, precision, recall, f1_score = score(y_true, pred)

                loss_list.append(loss)
                accuracy_list.append(accuracy)
                precision_list.append(precision)
                recall_list.append(recall)
                f1_list.append(f1_score)

            # Calculate the performance of each epoch and save it
            val_loss = np.array(loss_list).mean()
            val_accuracy = np.array(accuracy_list).mean()
            val_precision = np.array(precision_list).mean()
            val_recall = np.array(recall_list).mean()
            val_f1 = np.array(f1_list).mean()


        if val_f1 > val_best_f1:
            ## val result
            val_best_f1 = val_f1
            val_best_loss = val_loss
            val_best_accuracy = val_accuracy
            val_best_precision = val_precision
            val_best_recall = val_recall
            ## train result
            train_best_f1 = train_f1
            train_best_loss = train_loss
            train_best_accuracy = train_accuracy
            train_best_precision = train_precision
            train_best_recall = train_recall
            ## other result
            best_model = model
            iteration_num = 0
            best_pred_list = pred_list
        else:
            iteration_num += 1

        epoch_num += 1
        # Print out the results of each Epoch
        print(f'Epochs: {epoch_num + 1} || Train Loss: {train_loss: .3f}, Accuracy: {train_accuracy: .3f}, Precision: {train_precision: .3f}, Recall:{train_recall: .3f}, F1: {train_f1: .3f} || Val Loss: {val_loss: .3f}, Accuracy: {val_accuracy: .3f}, Precision: {val_precision: .3f}, Recall:{val_recall: .3f}, Val F1: {val_f1: .3f} || Best F1: {val_best_f1: .3f}, Iter: {iteration_num: .0f}')

    train_result = [train_best_loss, train_best_accuracy, train_best_precision,
                    train_best_recall, train_best_f1]
    val_result = [val_best_loss, val_best_accuracy, val_best_precision,
                  val_best_recall, val_best_f1]

    return model, train_result, val_result, best_pred_list

In [ ]:
"""
def train(model, class_weight_1, class_weight_2, train_set, val_set, patience=1000): #patience+


    train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)

    # Loss Function
    criterion_1 = nn.BCEWithLogitsLoss(pos_weight=class_weight_1)
    criterion_2 = nn.BCEWithLogitsLoss(pos_weight=class_weight_2) #


    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

    # Putting the model on the GPU to run
    model = model.to(device)
    criterion = criterion.to(device)

    val_best_f1 = float(-np.inf)
    iteration_num = 0
    epoch_num = 0

    epochs = config["epochs"]

    while True:

        if iteration_num >= patience:
            break


        # Train and save training results for ESG
        loss_list_1 = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        f1_list = []

        # Train and save training results for ESG items
        loss_list = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        f1_list = []



        # Set the model to the "training" state (Layers such as Dropout will be triggered only)
        model.train()

        for train_embeddings, train_labels_1, train_labels_2 in tqdm(train_loader):

            optimizer.zero_grad()

            train_embeddings = train_embeddings.to(device)
            train_labels_1 = train_labels_1.to(device)
            train_labels_2 = train_labels_2.to(device)

            output_1, output_2 = model(train_embeddings)

            batch_loss_1 = criterion_1(output_1, train_labels_1.float())
            batch_loss_2 = criterion_2(output_2, train_labels_2.float())
            batch_loss = batch_loss_1 + batch_loss_2  # 兩個loss會一起更改share_layer(black_block)

            # Back propagation
            batch_loss.backward()
            optimizer.step()


            # double
            output = output.cpu().detach()
            pred = (output > 0).cpu()
            y_true = (train_labels > 0).cpu()



            # Calculate the performance of each batch and save it
            # loss 可以都印出來(ESG/ESG items)
            loss = batch_loss.item()
            accuracy, precision, recall, f1_score= score(y_true, pred)

            loss_list.append(loss)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            recall_list.append(recall)
            f1_list.append(f1_score)

        # Calculate the performance of each epoch and save it
        train_loss = np.array(loss_list).mean()
        train_accuracy = np.array(accuracy_list).mean()
        train_precision = np.array(precision_list).mean()
        train_recall = np.array(recall_list).mean()
        train_f1 = np.array(f1_list).mean()


        # Evaluate the current model performance using validate dataset
        loss_list = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        pred_list = []
        f1_list = []

        with torch.no_grad():
            model.eval()
            for val_embeddings, val_labels in tqdm(val_loader):

                val_embeddings = val_embeddings.to(device)
                val_labels = val_labels.to(device)

                output = model(val_embeddings)

                batch_loss = criterion(output, val_labels.float())

                output = output.cpu()
                pred = (output > 0).cpu()
                sum_pred = torch.sum(pred, dim=1)
                non_label_case_idx = (sum_pred < 1).nonzero()

                if non_label_case_idx.shape[0] != 0:
                    non_label_case = output[non_label_case_idx]
                    max_col = torch.argmax(non_label_case, dim=-1)
                    raw = non_label_case_idx.reshape(-1,1)
                    pred_np = pred.numpy()
                    for idx in range(raw.shape[0]):
                        pred_np[raw[idx]][max_col[idx]] = True
                    pred = torch.from_numpy(pred_np)
                pred_list.extend(pred.numpy().astype('float32'))
                y_true = (val_labels > 0).cpu()

                # Calculate the performance of each batch and save it
                loss = batch_loss.item()
                accuracy, precision, recall, f1_score = score(y_true, pred)

                loss_list.append(loss)
                accuracy_list.append(accuracy)
                precision_list.append(precision)
                recall_list.append(recall)
                f1_list.append(f1_score)

            # Calculate the performance of each epoch and save it
            val_loss = np.array(loss_list).mean()
            val_accuracy = np.array(accuracy_list).mean()
            val_precision = np.array(precision_list).mean()
            val_recall = np.array(recall_list).mean()
            val_f1 = np.array(f1_list).mean()


        if val_f1 > val_best_f1:
            ## val result
            val_best_f1 = val_f1
            val_best_loss = val_loss
            val_best_accuracy = val_accuracy
            val_best_precision = val_precision
            val_best_recall = val_recall
            ## train result
            train_best_f1 = train_f1
            train_best_loss = train_loss
            train_best_accuracy = train_accuracy
            train_best_precision = train_precision
            train_best_recall = train_recall
            ## other result
            best_model = model
            iteration_num = 0
            best_pred_list = pred_list
        else:
            iteration_num += 1

        epoch_num += 1
        # Print out the results of each Epoch
        print(f'Epochs: {epoch_num + 1} || Train Loss: {train_loss: .3f}, Accuracy: {train_accuracy: .3f}, Precision: {train_precision: .3f}, Recall:{train_recall: .3f}, F1: {train_f1: .3f} || Val Loss: {val_loss: .3f}, Accuracy: {val_accuracy: .3f}, Precision: {val_precision: .3f}, Recall:{val_recall: .3f}, Val F1: {val_f1: .3f} || Best F1: {val_best_f1: .3f}, Iter: {iteration_num: .0f}')

    train_result = [train_best_loss, train_best_accuracy, train_best_precision,
                    train_best_recall, train_best_f1]
    val_result = [val_best_loss, val_best_accuracy, val_best_precision,
                  val_best_recall, val_best_f1]

    return model, train_result, val_result, best_pred_list
"""

In [ ]:
"""
# positive/negative
def train(model, class_weight_1, class_weight_2, class_weight_3, train_set, val_set, patience=1000): #patience+


    train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)

    # Loss Function
    criterion_1 = nn.BCEWithLogitsLoss(pos_weight=class_weight_1)
    criterion_2 = nn.BCEWithLogitsLoss(pos_weight=class_weight_2) #
    criterion_3 = nn.CrossEntropyLoss(weight=class_weight_3)

    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

    # Putting the model on the GPU to run
    model = model.to(device)
    criterion = criterion.to(device)

    val_best_f1 = float(-np.inf)
    iteration_num = 0
    epoch_num = 0

    epochs = config["epochs"]

    while True:

        if iteration_num >= patience:
            break


        # Train and save training results for ESG
        loss_list_1 = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        f1_list = []

        # Train and save training results for ESG items
        loss_list = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        f1_list = []



        # Set the model to the "training" state (Layers such as Dropout will be triggered only)
        model.train()

        for train_embeddings, train_labels_1, train_labels_2 in tqdm(train_loader):

            optimizer.zero_grad()

            train_embeddings = train_embeddings.to(device)
            train_labels_1 = train_labels_1.to(device)
            train_labels_2 = train_labels_2.to(device)

            output_1, output_2 = model(train_embeddings)

            batch_loss_1 = criterion_1(output_1, train_labels_1.float())
            batch_loss_2 = criterion_2(output_2, train_labels_2.float())
            batch_loss = batch_loss_1 + batch_loss_2  # 兩個loss會一起更改share_layer(black_block)

            # Back propagation
            batch_loss.backward()
            optimizer.step()


            # double
            output = output.cpu().detach()
            pred = output.argmax(dim=1).cpu()  # modified for multi class
            y_true = (train_labels > 0).cpu()



            # Calculate the performance of each batch and save it
            # loss 可以都印出來(ESG/ESG items)
            loss = batch_loss.item()
            accuracy, precision, recall, f1_score= score(y_true, pred)

            loss_list.append(loss)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            recall_list.append(recall)
            f1_list.append(f1_score)

        # Calculate the performance of each epoch and save it
        train_loss = np.array(loss_list).mean()
        train_accuracy = np.array(accuracy_list).mean()
        train_precision = np.array(precision_list).mean()
        train_recall = np.array(recall_list).mean()
        train_f1 = np.array(f1_list).mean()


        # Evaluate the current model performance using validate dataset
        loss_list = []
        accuracy_list = []
        precision_list = []
        recall_list = []
        pred_list = []
        f1_list = []

        with torch.no_grad():
            model.eval()
            for val_embeddings, val_labels in tqdm(val_loader):

                val_embeddings = val_embeddings.to(device)
                val_labels = val_labels.to(device)

                output = model(val_embeddings)

                batch_loss = criterion(output, val_labels.float())

                output = output.cpu()
                pred = (output > 0).cpu()
                sum_pred = torch.sum(pred, dim=1)
                non_label_case_idx = (sum_pred < 1).nonzero()

                if non_label_case_idx.shape[0] != 0:
                    non_label_case = output[non_label_case_idx]
                    max_col = torch.argmax(non_label_case, dim=-1)
                    raw = non_label_case_idx.reshape(-1,1)
                    pred_np = pred.numpy()
                    for idx in range(raw.shape[0]):
                        pred_np[raw[idx]][max_col[idx]] = True
                    pred = torch.from_numpy(pred_np)
                pred_list.extend(pred.numpy().astype('float32'))
                y_true = (val_labels > 0).cpu()

                # Calculate the performance of each batch and save it
                loss = batch_loss.item()
                accuracy, precision, recall, f1_score = score(y_true, pred)

                loss_list.append(loss)
                accuracy_list.append(accuracy)
                precision_list.append(precision)
                recall_list.append(recall)
                f1_list.append(f1_score)

            # Calculate the performance of each epoch and save it
            val_loss = np.array(loss_list).mean()
            val_accuracy = np.array(accuracy_list).mean()
            val_precision = np.array(precision_list).mean()
            val_recall = np.array(recall_list).mean()
            val_f1 = np.array(f1_list).mean()


        if val_f1 > val_best_f1:
            ## val result
            val_best_f1 = val_f1
            val_best_loss = val_loss
            val_best_accuracy = val_accuracy
            val_best_precision = val_precision
            val_best_recall = val_recall
            ## train result
            train_best_f1 = train_f1
            train_best_loss = train_loss
            train_best_accuracy = train_accuracy
            train_best_precision = train_precision
            train_best_recall = train_recall
            ## other result
            best_model = model
            iteration_num = 0
            best_pred_list = pred_list
        else:
            iteration_num += 1

        epoch_num += 1
        # Print out the results of each Epoch
        print(f'Epochs: {epoch_num + 1} || Train Loss: {train_loss: .3f}, Accuracy: {train_accuracy: .3f}, Precision: {train_precision: .3f}, Recall:{train_recall: .3f}, F1: {train_f1: .3f} || Val Loss: {val_loss: .3f}, Accuracy: {val_accuracy: .3f}, Precision: {val_precision: .3f}, Recall:{val_recall: .3f}, Val F1: {val_f1: .3f} || Best F1: {val_best_f1: .3f}, Iter: {iteration_num: .0f}')

    train_result = [train_best_loss, train_best_accuracy, train_best_precision,
                    train_best_recall, train_best_f1]
    val_result = [val_best_loss, val_best_accuracy, val_best_precision,
                  val_best_recall, val_best_f1]

    return model, train_result, val_result, best_pred_list
"""

### 10-Fold Cross-Validation

In [ ]:
def kfold_train_m_label(embedding_tensor, task_label, output_size, nfold=10):

    # 10 fold
    skf = StratifiedKFold(n_splits=nfold, shuffle=True, random_state=config["seed"])
    kfold_result = pd.DataFrame(columns=['Fold', 'train_loss', 'train_accuracy', 'train_precision',
                                    'train_recall', 'train_f1', 'val_loss', 'val_accuracy',
                                    'val_precision', 'val_recall', 'val_f1'])

    counts_true = pd.DataFrame(columns = ["num_label"])
    counts_pred = pd.DataFrame(columns = ["num_label"])

    pred_col_name = ["idx"]
    for o_idx in range(output_size):
        pred_col_name.append(f"{o_idx}")
    pred_result = pd.DataFrame(columns=pred_col_name)

    for fold_i, (train_fold, test_fold) in enumerate(skf.split(embedding_tensor, task_label.to_numpy().argmax(1))):

        print(f"Fold: {fold_i} | Train shape: {len(train_fold)} | Val shape: {len(test_fold)}")

        ## Preparation of training and validation datasets
        embedding_train, embedding_val = embedding_tensor[train_fold], embedding_tensor[test_fold]
        label_train, label_val = task_label.iloc[train_fold,:], task_label.iloc[test_fold,:]

        class_weight = print_class_weight(label_train)

        ## Actual number of labels
        actual_label_count = pd.DataFrame(sorted(Counter(label_val.sum(axis=1)).items()),
                                        columns = ["num_label", f"true_{fold_i}"])
        counts_true = pd.merge(counts_true, actual_label_count, how="outer", on = "num_label")

        ## Convert to Dataset data type
        train_set = Dataset(embedding_train, label_train)
        val_set = Dataset(embedding_val, label_val)

        print(f"Train 各類別資料量:", np.sum(train_set.__getLabels__(), axis=0))
        print(f"Val 各類別資料量:", np.sum(val_set.__getLabels__(), axis=0))
        model = Classifier(output_size=output_size)

        ## Model Training
        model, train_result, val_result, pred_list = train(model, class_weight, train_set, val_set)

        ## Record model evaluation results
        kfold_result = kfold_result.append([{"Fold": fold_i,
                                        "train_loss":train_result[0],
                                        "train_accuracy":train_result[1],
                                        "train_precision":train_result[2],
                                        "train_recall":train_result[3],
                                        "train_f1":train_result[4],
                                        "val_loss": val_result[0],
                                        "val_accuracy": val_result[1],
                                        "val_precision": val_result[2],
                                        "val_recall": val_result[3],
                                        "val_f1": val_result[4]}])


        ## Prediction results
        pred_list = np.array(pred_list)

        new_record = {}
        new_record['idx'] = test_fold

        for o_idx in range(1, len(pred_col_name)):
            new_record[pred_col_name[o_idx]] = pred_list[:,o_idx-1]

        pred_result = pred_result.append(pd.DataFrame(new_record), ignore_index=True)
        print("-"*100)

        ## Predicted number of labels
        pred_label_count = pd.DataFrame(sorted(Counter(pred_list.sum(axis=1)).items()),
                          columns=["num_label", f"pred_{fold_i}"])
        counts_pred = pd.merge(counts_pred, pred_label_count, how="outer", on = "num_label")


    return kfold_result, pred_result, counts_true, counts_pred

### Mainfunction

In [ ]:
kfold_result_DL, pred_result_DL, counts_true_DL, counts_pred_DL = kfold_train_m_label(embedding_tensor, label_data, label_data.shape[1])

### Evaluation

In [ ]:
pred_result_DL = pred_result_DL.sort_values("idx")
pred_result_DL.reset_index(inplace=True, drop=True)
display(kfold_result_DL)
display(kfold_result_DL.mean()[2:])